# Steam 게임 가격과 인기도 분석

`analysis.py`의 분석 흐름을 Jupyter Notebook 형태로 정리한 파일이다. 병합된 데이터(`data/merged_games.csv`)를 불러와 무료/유료 게임 비교, 가격과 긍정률 관계, 장르별 가성비 분석을 수행한다.

## 1. 라이브러리 및 경로 설정

In [1]:
from pathlib import Path
import os

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = ["Malgun Gothic", "DejaVu Sans", "Arial"]
plt.rcParams["axes.unicode_minus"] = False

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT.name == "src":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MERGED_CSV = DATA_DIR / "merged_games.csv"

OUTPUT_DIR.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT_DIR / ".matplotlib"))

MERGED_CSV

PosixPath('/content/data/merged_games.csv')

## 2. 데이터 불러오기 및 타입 변환

In [2]:
def load_dataset(path: Path = MERGED_CSV) -> pd.DataFrame:
    df = pd.read_csv(path)
    numeric_columns = [
        "price",
        "owners_avg",
        "positive_rate",
        "average_forever",
        "review_count",
        "value_score",
    ]
    for column in numeric_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")
    df["is_free"] = df["is_free"].astype(str).str.lower().isin(["true", "1", "yes"])
    return df.dropna(subset=["price", "owners_avg", "positive_rate", "genres"])


df = load_dataset()
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/data/merged_games.csv'

In [ ]:
df.info()
df.isna().sum()

## 3. 무료 게임과 유료 게임 비교

In [ ]:
free_paid_summary = (
    df.assign(type=df["is_free"].map({True: "Free", False: "Paid"}))
    .groupby("type", as_index=False)
    .agg(
        game_count=("appid", "count"),
        mean_owners=("owners_avg", "mean"),
        mean_positive_rate=("positive_rate", "mean"),
        mean_price=("price", "mean"),
    )
)

free_paid_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

sns.barplot(data=free_paid_summary, x="type", y="mean_owners", ax=axes[0])
axes[0].set_title("Average Owners")
axes[0].set_xlabel("")
axes[0].set_ylabel("Owners estimate")

sns.barplot(data=free_paid_summary, x="type", y="mean_positive_rate", ax=axes[1])
axes[1].set_title("Average Positive Rate")
axes[1].set_xlabel("")
axes[1].set_ylabel("Positive rate (%)")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "chart_free_vs_paid.png", dpi=160)
plt.show()

## 4. 가격과 사용자 평가의 관계

In [ ]:
paid = df[(df["price"] > 0) & df["positive_rate"].notna()].copy()
price_corr = paid[["price", "positive_rate"]].corr().iloc[0, 1]
print(f"Price/positive-rate correlation: {price_corr:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.regplot(
    data=paid,
    x="price",
    y="positive_rate",
    scatter_kws={"alpha": 0.35, "s": 20},
    line_kws={"color": "#d62728"},
    ax=ax,
)
ax.set_title("Price vs Positive Rate")
ax.set_xlabel("Price (KRW)")
ax.set_ylabel("Positive rate (%)")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "chart_price_vs_rating.png", dpi=160)
plt.show()

## 5. 장르별 가성비 분석

In [ ]:
def split_genres(value) -> list[str]:
    if value is None:
        return []
    text = str(value).strip()
    if not text or text.lower() == "nan":
        return []
    return [genre.strip() for genre in text.split(";") if genre.strip()]


def genre_frame(source_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in source_df.iterrows():
        for genre in split_genres(row["genres"]):
            rows.append(
                {
                    "genre": genre,
                    "price": row["price"],
                    "positive_rate": row["positive_rate"],
                    "average_forever": row["average_forever"],
                    "value_score": row["value_score"],
                    "review_count": row["review_count"],
                }
            )
    return pd.DataFrame(rows)


genres = genre_frame(df)
paid_genres = genres.dropna(subset=["value_score"])

genre_summary = (
    paid_genres.groupby("genre", as_index=False)
    .agg(
        game_count=("genre", "size"),
        mean_price=("price", "mean"),
        mean_positive_rate=("positive_rate", "mean"),
        mean_playtime=("average_forever", "mean"),
        mean_value_score=("value_score", "mean"),
        median_value_score=("value_score", "median"),
    )
    .query("game_count >= 5")
    .sort_values("median_value_score", ascending=False)
    .head(12)
)

genre_summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=genre_summary, y="genre", x="median_value_score", ax=ax)
ax.set_title("Top Genres by Median Value Score")
ax.set_xlabel("Median value score")
ax.set_ylabel("")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "chart_genre_value.png", dpi=160)
plt.show()

In [ ]:
heatmap_data = genre_summary.set_index("genre")[[
    "mean_price",
    "mean_positive_rate",
    "mean_playtime",
    "median_value_score",
]]

scaled = (heatmap_data - heatmap_data.mean()) / heatmap_data.std(ddof=0).replace(0, 1)

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(scaled, cmap="viridis", annot=False, ax=ax)
ax.set_title("Genre Metrics Heatmap (standardized)")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "chart_genre_heatmap.png", dpi=160)
plt.show()

## 6. 결과 요약

In [ ]:
print(f"분석 데이터 행 수: {len(df):,}")
print(f"무료 게임 수: {int(df['is_free'].sum()):,}")
print(f"유료 게임 수: {int((~df['is_free']).sum()):,}")
print(f"가격과 긍정률 상관계수: {price_corr:.4f}")
print(f"차트 저장 위치: {OUTPUT_DIR}")